# Máquinas Térmicas - Lección 6
## Cinética de Reacción

**Autor:** Camilo Bayona
**Fecha:** 08/08/2025

*Continuación de la Lección 5 (Repaso de Química): si allí vimos **qué** enlaces se rompen y forman, aquí estudiamos **a qué velocidad**.*

### Objetivos de aprendizaje
1. Definir la **tasa de reacción** y la **ley de velocidad**, distinguiendo orden de coeficiente estequiométrico.
2. Explicar la dependencia con la temperatura mediante la **ecuación de Arrhenius** y su base microscópica (**distribución de Maxwell-Boltzmann** y energía de activación).
3. Interpretar el **diagrama de coordenada de reacción** (energía de activación, entalpía, catálisis).
4. Derivar e integrar las **leyes de velocidad** de orden 0, 1 y 2, y calcular la **vida media**.


In [1]:
# Instalación solo en Google Colab; en el entorno local del curso es un no-op
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "matplotlib", "ipywidgets", "numpy", "scipy"], check=True)


In [2]:
# === Setup =================================================================
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown
from scipy.integrate import quad

R = 8.314  # J/(mol·K)
print("Setup OK — R =", R, "J/(mol·K)")


Setup OK — R = 8.314 J/(mol·K)


# Cinética de Reacción

La cinética química estudia la velocidad a la que ocurren las reacciones químicas y los factores que influyen en dichas velocidades. En este repaso se analizan los conceptos fundamentales, tales como la tasa de reacción, la ley de velocidad, la dependencia de la temperatura (Ecuación de Arrhenius) y la representación gráfica de una reacción.

## 1. Tasa de Reacción

La **tasa de reacción** se define como el cambio de concentración de un reactivo (o producto) por unidad de tiempo. Para un reactivo A:

$
\text{Tasa} = -\frac{d[A]}{dt}
$

El signo negativo indica que la concentración de A disminuye a medida que avanza la reacción.

## 2. Ley de Velocidad

Para una reacción general:

$
aA + bB \rightarrow cC + dD
$

La ley de velocidad se expresa habitualmente como:

$
\text{Velocidad} = k [A]^m [B]^n
$

donde:
- $ k $ es la constante de velocidad.
- $ [A] $ y $ [B] $ son las concentraciones de los reactivos.
- $ m $ y $ n $ son los órdenes de reacción respecto a A y B, determinados experimentalmente (no necesariamente iguales a los coeficientes estequiométricos $ a $ y $ b $).

## 3. Ejemplo: Combustión del Metano

La combustión del metano se puede escribir como:

$
\text{CH}_4 + 2\,\text{O}_2 \rightarrow \text{CO}_2 + 2\,\text{H}_2\text{O}
$

La ley de velocidad para esta reacción (según un estudio experimental hipotético) podría ser:

$
\text{Velocidad} = k [\text{CH}_4]^m [\text{O}_2]^n
$

Por ejemplo, si se determina que la reacción es de primer orden respecto al metano ($ m = 1 $) y de segundo orden respecto al oxígeno ($ n = 2 $), la expresión queda:

$
\text{Velocidad} = k [\text{CH}_4][\text{O}_2]^2
$

## 4. Ecuación de Arrhenius

La constante de velocidad $ k $ depende de la temperatura según la ecuación de Arrhenius:

$
k = A \, e^{-E_a/(RT)}
$

donde:
- $ A $ es el factor pre-exponencial (o de frecuencia).
- $ E_a $ es la energía de activación.
- $ R $ es la constante de los gases (8.314 J/(mol·K)).
- $ T $ es la temperatura en Kelvin.

Esta ecuación muestra que, al aumentar la temperatura, $ k $ aumenta, acelerando la reacción.

## 5. Representación Gráfica de la Cinética de Reacción

A continuación se muestran diagramas en Graphviz que representan el proceso de la combustión del metano, desde los reactivos pasando por el estado de transición (complejo activado) hasta los productos, e incluyen referencias a las constantes de velocidad y la ecuación de Arrhenius.




## Diagrama de coordenada de reacción
La **energía de activación** $E_a$ es la barrera que separa reactivos de productos: solo las colisiones con energía suficiente para alcanzar el **estado de transición** reaccionan. La diferencia de energía entre productos y reactivos es la **entalpía de reacción** $\Delta H$ (exotérmica si $\Delta H<0$). Un **catalizador** abre una ruta con menor $E_a$ sin alterar $\Delta H$.


In [3]:
# WIDGET — perfil de energía a lo largo de la coordenada de reacción
def coordenada_reaccion(Ea_kJ=80.0, dH_kJ=-40.0, catalizador=False):
    x = np.linspace(0, 1, 300)
    Ea = Ea_kJ; dH = dH_kJ
    def smooth(u):                                  # smoothstep 0->1 monótono
        u = np.clip(u, 0, 1); return 3 * u**2 - 2 * u**3
    def perfil(Ea_b):
        # dos tramos: reactivos(0) -> estado de transición(Ea_b) -> productos(dH)
        subida = Ea_b * smooth(x / 0.5)                       # 0 -> Ea_b
        bajada = Ea_b + (dH - Ea_b) * smooth((x - 0.5) / 0.5)  # Ea_b -> dH
        return np.where(x < 0.5, subida, bajada)
    fig, ax = plt.subplots(figsize=(8.5, 5))
    ax.plot(x, perfil(Ea), color="#2554E8", lw=2.5, label=f"sin catalizador ($E_a$={Ea:.0f})")
    if catalizador:
        Ea_cat = Ea * 0.55
        ax.plot(x, perfil(Ea_cat), color="#27AE60", lw=2.5, ls="--",
                label=f"con catalizador ($E_a$={Ea_cat:.0f})")
    ax.axhline(0, color="#888", lw=0.8); ax.axhline(dH, color="#888", lw=0.8, ls=":")
    ax.annotate("", xy=(0.5, Ea), xytext=(0.5, 0),
                arrowprops=dict(arrowstyle="<->", color="#C0392B", lw=1.5))
    ax.text(0.52, Ea / 2, f"$E_a$ = {Ea:.0f} kJ/mol", color="#C0392B", fontsize=10)
    ax.annotate("", xy=(0.92, dH), xytext=(0.92, 0),
                arrowprops=dict(arrowstyle="<->", color="#6C3483", lw=1.5))
    ax.text(0.72, dH / 2, f"$\\Delta H$ = {dH:.0f} kJ/mol", color="#6C3483", fontsize=10)
    ax.text(0.02, 3, "reactivos", fontsize=9); ax.text(0.85, dH + 3, "productos", fontsize=9)
    tipo = "exotérmica" if dH < 0 else "endotérmica"
    ax.set_xlabel("coordenada de reacción"); ax.set_ylabel("energía [kJ/mol]")
    ax.set_title(f"Perfil energético — reacción {tipo}")
    ax.legend(loc="upper right"); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

interact(coordenada_reaccion,
         Ea_kJ=FloatSlider(80, min=20, max=200, step=5, description="Eₐ [kJ/mol]"),
         dH_kJ=FloatSlider(-40, min=-150, max=100, step=10, description="ΔH [kJ/mol]"),
         catalizador=Dropdown(options=[("no", False), ("sí", True)], value=False, description="catalizador"));


interactive(children=(FloatSlider(value=80.0, description='Eₐ [kJ/mol]', max=200.0, min=20.0, step=5.0), Float…

## Base microscópica: distribución de Maxwell-Boltzmann
La temperatura no cambia $E_a$: cambia **cuántas moléculas la superan**. La fracción de moléculas con energía mayor que $E_a$ crece exponencialmente con $T$, y esa fracción es precisamente el factor $e^{-E_a/RT}$ de Arrhenius. El widget sombrea el área de la cola con $E>E_a$.


In [4]:
# WIDGET — distribución de energía de Maxwell-Boltzmann y fracción con E > Ea
def maxwell_boltzmann(T=500.0, Ea_kJ=50.0):
    Ea = Ea_kJ * 1e3                     # J/mol
    E = np.linspace(0, max(Ea * 2.5, 1.2e5), 500)
    f = 2 * np.sqrt(E / np.pi) * (1 / (R * T)) ** 1.5 * np.exp(-E / (R * T))
    frac = np.exp(-Ea / (R * T))         # fracción aproximada con E > Ea
    fig, ax = plt.subplots(figsize=(8.5, 5))
    ax.plot(E / 1e3, f, color="#2554E8", lw=2, label=f"T = {T:.0f} K")
    ax.fill_between(E / 1e3, 0, f, where=(E >= Ea), color="#C0392B", alpha=0.4,
                    label=f"E > Eₐ  (fracción ≈ {frac:.2e})")
    # curva a mayor temperatura para comparar
    T2 = T + 300
    f2 = 2 * np.sqrt(E / np.pi) * (1 / (R * T2)) ** 1.5 * np.exp(-E / (R * T2))
    ax.plot(E / 1e3, f2, color="#E8A000", lw=1.5, ls="--", label=f"T = {T2:.0f} K")
    ax.axvline(Ea / 1e3, color="#C0392B", lw=1.5)
    ax.text(Ea / 1e3, ax.get_ylim()[1] * 0.9, " Eₐ", color="#C0392B", fontsize=11)
    ax.set_xlabel("Energía molecular [kJ/mol]"); ax.set_ylabel("fracción de moléculas  f(E)")
    ax.set_title("Distribución de Maxwell-Boltzmann: solo la cola con E > Eₐ reacciona")
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
    print(f"A T = {T:.0f} K, fracción con E > Eₐ ≈ {frac:.3e}. "
          f"Subir 300 K la multiplica por {np.exp(-Ea/(R*T2))/frac:.1f}×.")

interact(maxwell_boltzmann,
         T=FloatSlider(500, min=200, max=2000, step=50, description="T [K]"),
         Ea_kJ=FloatSlider(50, min=10, max=150, step=5, description="Eₐ [kJ/mol]"));


interactive(children=(FloatSlider(value=500.0, description='T [K]', max=2000.0, min=200.0, step=50.0), FloatSl…

## Ecuación de Arrhenius: $k(T)=A\,e^{-E_a/RT}$
En forma linealizada, $\ln k = \ln A - \dfrac{E_a}{R}\,\dfrac{1}{T}$: una recta de pendiente $-E_a/R$ en el plano $\ln k$ vs $1/T$, de donde se mide experimentalmente $E_a$.


In [5]:
# WIDGET — Arrhenius: k(T) y su linealización ln k vs 1/T
def arrhenius(Ea_kJ=75.0, logA=12.0):
    A = 10.0 ** logA; Ea = Ea_kJ * 1e3
    T = np.linspace(300, 1500, 300)
    k = A * np.exp(-Ea / (R * T))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    ax1.plot(T, k, color="#2554E8", lw=2)
    ax1.set_yscale("log"); ax1.set_xlabel("Temperatura [K]")
    ax1.set_ylabel("k [s⁻¹]"); ax1.set_title("k(T) — escala log")
    ax1.grid(alpha=0.3, which="both")
    ax2.plot(1 / T, np.log(k), color="#C0392B", lw=2)
    ax2.set_xlabel("1/T [K⁻¹]"); ax2.set_ylabel("ln k")
    ax2.set_title(f"Linealización: pendiente = −Eₐ/R = {-Ea/R:.0f} K")
    ax2.grid(alpha=0.3)
    pend = np.polyfit(1 / T, np.log(k), 1)[0]
    ax2.text(0.05, 0.1, f"Eₐ recuperada = {-pend*R/1e3:.1f} kJ/mol",
             transform=ax2.transAxes, bbox=dict(boxstyle="round", fc="wheat"))
    fig.suptitle(f"Arrhenius: A = 10^{logA:.0f} s⁻¹, Eₐ = {Ea_kJ:.0f} kJ/mol", fontsize=11)
    plt.tight_layout(); plt.show()

interact(arrhenius,
         Ea_kJ=FloatSlider(75, min=20, max=200, step=5, description="Eₐ [kJ/mol]"),
         logA=FloatSlider(12, min=6, max=16, step=0.5, description="log₁₀A"));


interactive(children=(FloatSlider(value=75.0, description='Eₐ [kJ/mol]', max=200.0, min=20.0, step=5.0), Float…

# Derivación de la Ecuación de Concentración en Función del Tiempo

Consideremos una reacción de primer orden, donde la tasa de disminución de la concentración es proporcional a la concentración misma. La ecuación diferencial es:

$
\frac{dC}{dt} = -k C,
$

donde:
- $ C(t) $ es la concentración en el tiempo $ t $.
- $ k $ es la constante de velocidad de la reacción.

## 1. Separación de Variables

Reorganizamos la ecuación para separar las variables $ C $ y $ t $:

$
\frac{dC}{C} = -k \, dt.
$

## 2. Integración

Integramos ambos lados de la ecuación:

$
\int \frac{1}{C}\, dC = -k \int dt.
$

La integral del lado izquierdo es:

$
\int \frac{1}{C}\, dC = \ln |C| + C_1,
$

y la integral del lado derecho es:

$
\int dt = t + C_2.
$

Podemos combinar las constantes de integración ($ C_1 $ y $ C_2 $) en una única constante. Escribimos:

$
\ln |C| = -k t + \ln C_0,
$

donde $ \ln C_0 $ es la constante de integración determinada por la condición inicial $ C(0) = C_0 $.

## 3. Resolviendo para $ C(t) $

Exponenciamos ambos lados de la ecuación para despejar $ C $:

$
|C| = e^{-k t + \ln C_0} = C_0 \, e^{-k t}.
$

Como la concentración $ C $ es siempre positiva, podemos omitir el valor absoluto:

$
C(t) = C_0 \, e^{-k t}.
$

## Resultado Final

La concentración en función del tiempo para una reacción de primer orden se expresa como:

$
\boxed{C(t) = C_0 \, e^{-k t}},
$

donde:
- $ C_0 $ es la concentración inicial.
- $ k $ es la constante de velocidad.
- $ t $ es el tiempo.


## Leyes de velocidad integradas: orden 0, 1 y 2
La forma en que $[A]$ decae en el tiempo revela el **orden** de la reacción:

| Orden | Ley integrada | Recta característica | Vida media $t_{1/2}$ |
|---|---|---|---|
| 0 | $[A]=[A]_0-kt$ | $[A]$ vs $t$ | $[A]_0/2k$ |
| 1 | $[A]=[A]_0e^{-kt}$ | $\ln[A]$ vs $t$ | $\ln 2/k$ |
| 2 | $1/[A]=1/[A]_0+kt$ | $1/[A]$ vs $t$ | $1/(k[A]_0)$ |


In [6]:
# WIDGET — decaimiento de la concentración según el orden de reacción
def concentracion(orden="1", k=0.5, C0=1.0):
    t = np.linspace(0, 12, 300)
    if orden == "0":
        C = np.maximum(C0 - k * t, 0); t12 = C0 / (2 * k)
    elif orden == "1":
        C = C0 * np.exp(-k * t); t12 = np.log(2) / k
    else:
        C = C0 / (1 + k * C0 * t); t12 = 1 / (k * C0)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))
    ax1.plot(t, C, color="#2554E8", lw=2)
    ax1.axhline(C0 / 2, ls=":", color="#888"); ax1.axvline(t12, ls=":", color="#C0392B")
    ax1.text(t12, C0 * 0.6, f" t½ = {t12:.2f} s", color="#C0392B", fontsize=10)
    ax1.set_xlabel("t [s]"); ax1.set_ylabel("[A]"); ax1.set_title(f"Concentración (orden {orden})")
    ax1.grid(alpha=0.3); ax1.set_ylim(0, C0 * 1.05)
    # recta característica que linealiza ese orden
    with np.errstate(divide="ignore"):
        if orden == "0":
            y = C; lab = "[A]"
        elif orden == "1":
            y = np.log(np.maximum(C, 1e-9)); lab = "ln[A]"
        else:
            y = 1 / np.maximum(C, 1e-9); lab = "1/[A]"
    ax2.plot(t, y, color="#27AE60", lw=2)
    ax2.set_xlabel("t [s]"); ax2.set_ylabel(lab)
    ax2.set_title(f"Recta característica del orden {orden}: {lab} vs t"); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

interact(concentracion,
         orden=Dropdown(options=["0", "1", "2"], value="1", description="orden"),
         k=FloatSlider(0.5, min=0.05, max=2.0, step=0.05, description="k"),
         C0=FloatSlider(1.0, min=0.2, max=3.0, step=0.1, description="[A]₀"));


interactive(children=(Dropdown(description='orden', index=1, options=('0', '1', '2'), value='1'), FloatSlider(…

## Conclusión
La cinética conecta la química con la ingeniería térmica: la **velocidad** de combustión, la formación de contaminantes y la estabilidad de un combustible dependen de $E_a$, del factor $A$ y de la temperatura a través de Arrhenius. Un aumento de temperatura desplaza la cola de Maxwell-Boltzmann y acelera exponencialmente la reacción; un catalizador baja $E_a$ sin cambiar la termodinámica. Estas herramientas se aplican en las lecciones de **combustión** y **motores**.

---
### Referencias
- Atkins, P. & de Paula, J. (2018). *Physical Chemistry*, 11ª ed. Oxford — cinética química.
- Levenspiel, O. (1999). *Chemical Reaction Engineering*, 3ª ed. Wiley.
- Turns, S. R. (2012). *An Introduction to Combustion*, 3ª ed. McGraw-Hill.
